# 融合算子优化与 Profiling

第 3 章我们用基线 SFT 配置训练了 Wordle 模型，loss 从 1.25 降至 0.49，format_reward 达到 1.0。

但训练性能还有优化空间。本章从性能瓶颈分析出发，引入 Ascend NPU 融合算子，通过 ModelConverter 一行配置接入，最后用 profiling 数据验证优化效果。

---

## 教程进度回顾

| 章节 | 内容 | 状态 |
|------|------|------|
| 第 1 章 | SFT 概念 + Wordle 任务 | ✅ 已完成 |
| 第 2 章 | TorchTitan 框架 + 环境配置 | ✅ 已完成 |
| 第 3 章 | 数据准备 + 基线训练 + 推理评测 | ✅ 已完成 |
| **第 4 章** | **融合算子 + Profiling 优化** | ← 当前 |

---

## 基线性能回顾：瓶颈在哪？

回顾第 3 章的基线训练数据（Qwen3-1.7B，单卡 Ascend）：

| 指标 | 值 | 说明 |
|------|-----|------|
| 吞吐量 | ~5,600 tps | tokens per second |
| MFU | ~16.5% | 模型计算利用率偏低 |
| Memory/Compute | ~2.89 | 模型整体 **memory-bound** |

> **MFU 16.5% 的原因**：小模型（1.7B）在单卡上受限于**内存带宽**而非计算能力。大量时间花在等待 HBM 读写上，而非实际计算。

### 算子碎片化

以 **RMSNorm** 为例：Qwen3-1.7B 有 28 层 × 2 个 RMSNorm = 56 个实例。每个原生 PyTorch RMSNorm 被展开为 6 个独立 kernel，共 **336 次微小 kernel 发射/step**。每次发射有 10–15 μs 的 host→device 提交延迟，累积开销显著。同时每个小 kernel 独立读写 HBM，带宽利用率低。

---

## 解决方案

用 Ascend NPU 融合算子 `torch_npu.npu_rms_norm` 将 6 个 kernel 合并为 1 个 —— 具体原理和实现在 04.02 详述。本章随后跑 profiling 对比验证实际收益。

---

## 前置条件

- 第 2 章环境已配置，`torch_npu` 可正常 `import`。
- 了解第 2 章 ModelConverter 的基本概念（注册 → 配置 → 转换）。

---

## 本章结构

| Notebook | 内容 |
|---|---|
| [04.01](04.01_chapter_intro.ipynb) | 章节介绍（本节） |
| [04.02](04.02_adding_fused_operator.ipynb) | 融合算子实现：`rms_norm.py` 代码详解 |
| [04.03](04.03_running_and_profiling.ipynb) | Profiling 实操：跑 baseline vs fused |
| [04.04](04.04_profiling_output_analysis.ipynb) | Profiling 结果分析 |
| [04.05](04.05_chapter_practice.ipynb) | 章节练习 |

下一节，我们将深入 `fused_operators/rms_norm.py` 的每一行代码。

## 练习

1. （判断题）Qwen3-1.7B 每层有 2 个 RMSNorm（attention_norm + ffn_norm），28 层共 56 个 RMSNorm 实例。

2. （判断题）算子碎片化的主要代价是模型参数量增加和显存占用升高。

3. （单选题）以下哪项最能描述第 3 章基线训练中 MFU 只有 16.5% 的原因？
    A. NPU 计算单元太少
    B. 模型太小，受限于内存带宽（memory-bound）而非计算能力
    C. 学习率设置过低
    D. 数据加载速度太慢

In [ ]:
!cat ./answer/04.01_answer.txt